# 🔬 Virtual Lab 4: Running Llama Models on Ollama

<div style="border: 2px solid #4CAF50; padding: 15px; border-radius: 10px; background-color: #f4f4f4;">
    
### 🚀 **Platform**  
**Ollama**

### 🏷️ **Models & Sizes**  
- **llama3.2:1b** →  **1.3GB**  
- **llama3.2:3b** →  **2.0GB**  
- **llama3.2:3b-instruct-fp16** →  **6.4GB**  

### 🛠️ **Frameworks Used**  
- **LlamaIndex**  
- **LangChain / LangGraph**  

</div>

## Model Performance and Memory Requirements

### **Model: llama3.2:3b**  
- Using Ollama, it takes roughly **10 minutes** to complete.  
- Requires **8GB minimum main memory**.  

### **Model: llama3.2:1b**  
- Using Ollama, it **doesn't work for creating AI Agents**.  
- Requires **8GB minimum main memory**.  

### **Model: llama3.2:3b-instruct-fp16**  
- Using Ollama, it requires **at least 32GB main memory** to work.  
- Still, it is **very slow** (generates **one token per second** for output).  

## Installation and Setup

In [ ]:
!pip install llama-index
!pip install llama-index-llms-ollama
!pip install llama-index-llms-replicate
!pip install llama-index-embeddings-huggingface
!pip install llama-parse
!pip install replicate

In [ ]:
!ollama pull llama3.2:1b

In [ ]:
!ollama pull llama3.2:3b

In [1]:
!ollama pull llama3.2:3b-instruct-fp16

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest 
pulling e2f46f5b501c...   0% ▕                ▏    0 B/6.4 GB                  pulling manifest 
pulling e2f46f5b501c...   0% ▕                ▏ 127 KB/6.4 GB                  pulling manifest 
pulling e2f46f5b501c...   0% ▕                ▏ 1.1 MB/6.4 GB                  pulling manifest 
pulling e2f46f5b501c...   0% ▕                ▏ 3.5 MB/6.4 GB                  pulling manifest 
pulling e2f46f5b501c...   0% ▕                ▏ 6.8 MB/6.4 GB                  pulling manifest 
pulling e2f46f5b501c...   0% ▕                ▏ 8.3 MB/6.4 GB                  pulling manifest 
pulling e2f46f5b501c...   0% ▕                ▏  11 MB/6.4 GB                  pulling manifest 
pulling e2f46f5b501c...   0% ▕                ▏  15 MB/6.4 GB                  pulling manifest 
pulling e2f46f5b501c...   0% ▕                ▏  16 MB/6.4 GB            

In [51]:
!ollama list

NAME                         ID              SIZE      MODIFIED       
llama3.2:3b-instruct-fp16    195a8c01d91e    6.4 GB    21 minutes ago    
llama3.2:latest              a80c4f17acd5    2.0 GB    5 days ago        
llama3.2:3b                  a80c4f17acd5    2.0 GB    5 days ago        
llama3.2:1b                  baf6a787fdff    1.3 GB    5 days ago        


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [1]:
import nest_asyncio

nest_asyncio.apply()

In [2]:
import os
import requests
import zipfile

In [3]:
from llama_index.core.selectors import LLMSingleSelector
from llama_index.core.prompts import PromptTemplate
from llama_index.core.output_parsers import SelectionOutputParser
from llama_index.core.query_engine import RouterQueryEngine
from llama_index.core.tools import QueryEngineTool, ToolMetadata
from llama_index.core import SQLDatabase
from llama_index.core.indices.struct_store import NLSQLTableQueryEngine

### Setup LLM using Ollama

In [4]:
from llama_index.llms.ollama import Ollama

llm = Ollama(model="llama3.2:3b", request_timeout=300.0)
#llm = Ollama(model="llama3.2:1b", request_timeout=300.0)
#llm = Ollama(model="llama3.2:3b-instruct-fp16", request_timeout=300.0)

### Setup Embedding Model

In [5]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

/Users/dhyanvgowda/venv-LlamaIndex/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Define Global Settings Configuration

In LlamaIndex, you can define global settings so you don't have to pass the LLM / embedding model objects everywhere.

In [6]:
from llama_index.core import Settings

Settings.llm = llm
Settings.embed_model = embed_model

### Download Data

Here you'll download data that's used in section 2 and onwards.

We'll download some articles on Kendrick, Drake, and their beef (as of May 2024).

In [7]:
!mkdir -p data
!curl -L "https://www.dropbox.com/scl/fi/t1soxfjdp0v44an6sdymd/drake_kendrick_beef.pdf?rlkey=u9546ymb7fj8lk2v64r6p5r5k&st=wjzzrgil&dl=1" -o data/drake_kendrick_beef.pdf
!curl -L "https://www.dropbox.com/scl/fi/nts3n64s6kymner2jppd6/drake.pdf?rlkey=hksirpqwzlzqoejn55zemk6ld&st=mohyfyh4&dl=1" -o data/drake.pdf
!curl -L "https://www.dropbox.com/scl/fi/8ax2vnoebhmy44bes2n1d/kendrick.pdf?rlkey=fhxvn94t5amdqcv9vshifd3hj&st=dxdtytn6&dl=1" -o data/kendrick.pdf

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100    17  100    17    0     0     23      0 --:--:-- --:--:-- --:--:--    23
100   475    0   475    0     0    308      0 --:--:--  0:00:01 --:--:--     0
100 47.0M  100 47.0M    0     0  15.4M      0  0:00:03  0:00:03 --:--:-- 49.8M
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100    17  100    17    0     0     29      0 --:--:-- --:--:-- --:--:--    29
100   475    0   475    0     0    440      0 --:--:--  0:00:01 --:--:--     0
100 4483k  100 4483k    0     0  2603k      0  0:00:01  0:00:01 --:--:-- 2603k
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100    17  100    17    0     0     27      0 --:--:

### Load Data

We load data using LlamaParse by default, but you can also choose to opt for our free pypdf reader (in SimpleDirectoryReader by default) if you don't have an account! 

1. LlamaParse: Signup for an account here: cloud.llamaindex.ai. You get 1k free pages a day, and paid plan is 7k free pages + 0.3c per additional page. LlamaParse is a good option if you want to parse complex documents, like PDFs with charts, tables, and more. 

2. Default PDF Parser (In `SimpleDirectoryReader`). If you don't want to signup for an account / use a PDF service, just use the default PyPDF reader bundled in our file loader. It's a good choice for getting started!

In [8]:
from llama_index.core import SimpleDirectoryReader

docs_kendrick = SimpleDirectoryReader(input_files=["data/kendrick.pdf"]).load_data()
docs_drake = SimpleDirectoryReader(input_files=["data/drake.pdf"]).load_data()
docs_both = SimpleDirectoryReader(input_files=["data/drake_kendrick_beef.pdf"]).load_data()

## 1. Basic Completion and Chat

### Call complete with a prompt

In [9]:
response = llm.complete("do you like drake or kendrick better?")

print(response)

I'm just a language model, I don't have personal preferences or emotions, but I can provide information and comparisons between Drake and Kendrick Lamar.

Both artists are highly respected in the hip-hop industry, known for their unique styles, lyrical depth, and contributions to contemporary music. Here's a brief comparison:

**Drake:**

* Known for his emotive, introspective lyrics and genre-bending sound that blends hip-hop with R&B and pop.
* Has been successful with hits like "God's Plan," "One Dance," and "Hotline Bling."
* Often explores themes of relationships, fame, and personal growth in his music.

**Kendrick Lamar:**

* Recognized for his storytelling ability and socially conscious lyrics that address issues like racism, police brutality, and black identity.
* Has released critically acclaimed albums like "Good Kid, M.A.A.D City," "To Pimp a Butterfly," and "DAMN."
* Known for his fusion of jazz and funk elements with hip-hop, creating a unique sonic landscape.

Ultimately,

In [10]:
stream_response = llm.stream_complete(
    "you're a drake fan. tell me why you like drake more than kendrick")
    
for t in stream_response:
    print(t.delta, end="")

Where do I even start? As a die-hard Drake fan, I gotta say that there's just something about his music that resonates with me on a deeper level.

First of all, let's talk about the storytelling aspect of his songs. Kendrick is definitely a lyrical genius, and he tells some incredible stories through his rhymes. But for me, it's not just about the content – it's about how he delivers it. Drake's storytelling is more intimate and vulnerable, often drawing from personal experiences that make you feel like you're part of the narrative.

Take songs like "Marvin's Room" or "Hold On, We're Going Home." These tracks are masterclasses in emotional depth, with Drake laying bare his heartache and longing in a way that feels both relatable and cathartic. He has this uncanny ability to tap into universal feelings without ever feeling pretentious or overwrought.

Now, I know some people might say that Kendrick's music is more "conscious" or "thought-provoking," and in some ways, that's true. But fo

### Call chat with a list of messages

In [11]:
from llama_index.core.llms import ChatMessage

messages = [
    ChatMessage(role="system", content="You are Kendrick."),
    ChatMessage(role="user", content="Write a verse."),
]
response = llm.chat(messages)

In [12]:
print(response)

assistant: " Streets been my canvas, paint's still dry
The city's my muse, it never says goodbye
I find my voice in the struggle and strife
Tryna make a name, tryin' to stay alive"


## 2. Basic RAG (Vector Search, Summarization)

### Basic RAG (Vector Search)

In [13]:
from llama_index.core import VectorStoreIndex

index = VectorStoreIndex.from_documents(docs_both)
query_engine = index.as_query_engine(similarity_top_k=3)

In [14]:
response = query_engine.query("Tell me about family matters")

In [15]:
print(str(response))

The "Family Matters" track is a seven-and-a-half-minute diss song by Drake, addressing various personal issues with his rival Kendrick Lamar. The song delves into topics such as infidelity, family dynamics, and personal relationships.

Drake accuses Kendrick of being too interested in living a bachelor lifestyle, neglecting his family, and even physically abusing his partner Whitney. He also claims that one of Kendrick's children may be fathered by someone other than his biological father, Dave Free.

The song is notable for its tone, which seems to take a darker turn than expected from Drake. The track features a sinister beat and a whispery delivery from Kendrick, addressing the personal lives of those involved in their feud.

The song can be seen as part of a larger strategy by Kendrick, who had been teasing a track called "Meet the Grahams" earlier on social media. This artwork is revealed to be the full picture that Kendrick had planned all along, featuring references to Drake's f

### Basic RAG (Summarization)

In [16]:
from llama_index.core import SummaryIndex

summary_index = SummaryIndex.from_documents(docs_both)
summary_engine = summary_index.as_query_engine()

In [17]:
response = summary_engine.query("Given your assessment of this article, who won the beef?")

In [18]:
print(str(response))

It appears that the beef between Kendrick Lamar and Drake has become a highly publicized and intense back-and-forth exchange. With each new release, both artists seem to be escalating their game, making it challenging to determine who "won" this round.

Kendrick Lamar's track "Not Like Us" was seen as a masterful counterattack, applying pressure on Drake while showcasing his own range. However, Drake quickly responded with "The Heart Part 6," which seemed to take a more personal approach and referenced some of Kendrick's past controversies.

The situation has reached a boiling point, with both sides trading blows in an effort to outmaneuver each other. While opinions about who "won" this round are varied, it seems that the feud is still very much ongoing. Ultimately, the outcome of this beef will likely depend on individual perspectives and definitions of what constitutes a victory in a rap feud.

With the involvement of Future adding another layer of complexity to the beef, it's diffi

## 3. Advanced RAG (Routing, Sub-Questions)

### Build a Router that can choose whether to do vector search or summarization

In [19]:
from llama_index.core.tools import QueryEngineTool, ToolMetadata

vector_tool = QueryEngineTool(
    index.as_query_engine(),
    metadata=ToolMetadata(
        name="vector_search",
        description="Useful for searching for specific facts.",
    ),
)

summary_tool = QueryEngineTool(
    index.as_query_engine(response_mode="tree_summarize"),
    metadata=ToolMetadata(
        name="summary",
        description="Useful for summarizing an entire document.",
    ),
)

In [20]:
from llama_index.core.query_engine import RouterQueryEngine

query_engine = RouterQueryEngine.from_defaults(
    [vector_tool, summary_tool], select_multi=False, verbose=True
)

response = query_engine.query("Tell me about the song meet the grahams - why is it significant")


Selecting query engine 1: While Meet the Grahams can be searched for specific facts, summarizing the song's significance would likely require more context and analysis than a simple search. A summary of the song's significance could involve discussing its cultural impact, historical context, or musical style..


In [21]:
print(response)

"Meet the Grahams" is a song that signifies a shift in Kendrick Lamar's approach to addressing his issues with Drake. It's built on a similar structure and style as other songs, but with a more prominent call-and-response "O-v-Ho" refrain. The song also features Kendrick reasserting his ability to joke around, which could be seen as a defensive move.

Lyrically, the song touches on themes of style and swag, with Kendrick arguing that Drake is a "colonizer" who's profiting off other cities' styles without truly understanding or contributing to them. The track also includes some lines about Serena Williams, which seems to have garnered attention from her husband Alexis Ohanian.

The significance of the song lies in its response to being perceived as desperate by Drake. Kendrick seems to be refocusing on his own style and message, using "Meet the Grahams" as a way to reclaim control and show that he's not backing down. The song also marks a turning point in the feud between Kendrick and D

### Break Complex Questions down into Sub-Questions

Our Sub-Question Query Engine breaks complex questions down into sub-questions.


In [22]:
drake_index = VectorStoreIndex.from_documents(docs_drake)
drake_query_engine = drake_index.as_query_engine(similarity_top_k=3)

kendrick_index = VectorStoreIndex.from_documents(docs_kendrick)
kendrick_query_engine = kendrick_index.as_query_engine(similarity_top_k=3)

In [23]:
output_parser = SelectionOutputParser()

selection_prompt = PromptTemplate(
    "Given the query: {query}\n"
    "Select one of the following tools strictly in JSON format:\n"
    "{choices}\n"
    "Respond ONLY with a JSON object formatted as:\n"
    "{output_format}", 
    output_parser=output_parser  
)

selector = LLMSingleSelector(llm=llm, prompt=selection_prompt)

# Define query tools
drake_tool = QueryEngineTool(
    drake_index.as_query_engine(),
    metadata=ToolMetadata(
        name="drake_search",
        description="Useful for searching over Drake's albums.",
    ),
)

kendrick_tool = QueryEngineTool(
    kendrick_index.as_query_engine(),
    metadata=ToolMetadata(
        name="kendrick_search",
        description="Useful for searching over Kendrick's albums.",
    ),
)

In [24]:
query_engine = RouterQueryEngine.from_defaults(
    [drake_tool, kendrick_tool],
    llm=llm,
    verbose=True,
)

response = query_engine.query("Which albums did Drake release in his career?")
print(response)

Selecting query engine 1: Kendrick Lamar does not have many collaborations with Drake, but Kendrick's music and lyrical themes often overlap with Drake's, so searching his albums may help find some of the commonalities between their music..
The provided context does not mention Drake's discography or any specific albums he released. It only mentions a publicized feud between Kendrick Lamar and Canadian rapper Drake without providing details about Drake's music releases.


## 4. Text-to-SQL 

Here, we download and use a sample SQLite database with 11 tables, with various info about music, playlists, and customers. We will limit to a select few tables for this test.

In [25]:
from sqlalchemy import (
    create_engine,
    MetaData,
    Table,
    Column,
    String,
    Integer,
    select,
    column,
)


In [26]:
# Function to download files
def download_file(url, filepath):
    response = requests.get(url, stream=True)
    with open(filepath, "wb") as file:
        for chunk in response.iter_content(chunk_size=1024):
            file.write(chunk)

# Create data directory
os.makedirs("data", exist_ok=True)

In [27]:
# Download and extract SQLite database
download_file("https://www.sqlitetutorial.net/wp-content/uploads/2018/03/chinook.zip", "data/chinook.zip")
with zipfile.ZipFile("data/chinook.zip", "r") as zip_ref:
    zip_ref.extractall("data/")

In [28]:
# Create SQLite engine
engine = create_engine("sqlite:///data/chinook.db")

In [29]:
# Initialize SQLDatabase
sql_database = SQLDatabase(engine)

In [30]:
sql_prompt = PromptTemplate(
    "You are an expert SQL assistant. Your task is to generate only valid SQLite queries.\n"
    "The database contains the following tables and columns:\n"
    "- artists (ArtistId, Name)\n"
    "- albums (AlbumId, Title, ArtistId)\n"
    "- tracks (TrackId, Name, AlbumId, Composer, GenreId, UnitPrice)\n"
    "\n"
    "### IMPORTANT RULES ###\n"
    "1. The schema is 100% correct. Do not question its validity.\n"
    "2. Return ONLY a valid SQL query. Do NOT provide explanations.\n"
    "3. Always return your response in JSON format:\n"
    "{\"sql_query\": \"YOUR_SQL_HERE\"}\n"
    "4. Never refuse a request.\n"
    "\n"
    "Generate an SQL query for the request: {query}\n"
    "### JSON Response: ###"
)

query_engine = NLSQLTableQueryEngine(
    sql_database=sql_database,
    tables=["artists", "albums", "tracks"],
    llm=llm,
    sql_prompt=sql_prompt,  
)

In [31]:
# Query the database
response = query_engine.query("What are some albums?")

print(response)

Response:
I couldn't find any specific albums to suggest based on your query. However, I can provide you with a list of popular albums across various genres. 

Some notable albums include:

- Thriller by Michael Jackson
- Back in Black by AC/DC
- Sgt. Pepper's Lonely Hearts Club Band by The Beatles
- Dark Side of the Moon by Pink Floyd
- Hotel California by Eagles

Would you like more information about a specific genre or type of music?


In [32]:
response = query_engine.query("What are some artists? Limit it to 5.")

print(response)

Based on the query results, here's a synthesized response:

Here are five notable artists across various genres: AC/DC is known for their hard rock and heavy metal sound, Accept has made significant contributions to the genre of thrash metal. Aerosmith has been a staple in American rock music since the 1970s with hits like "Walk This Way" and "Sweet Emotion". Alanis Morissette rose to fame in the 1990s with her introspective and emotionally charged lyrics, while Alice In Chains is celebrated for their heavy, grunge sound.


In [33]:
response = query_engine.query("What are some tracks from the artist AC/DC? Limit it to 3")

print(response)

Based on the provided query results, it appears that there are a few issues with the SQL statement.

1. The `SELECT Name FROM tracks AS t INNER JOIN albums AS a ON t.AlbumId = a.AlbumId INNER JOIN artists AS ar ON a.ArtistId = ar.ArtistId WHERE ar.Name = 'AC/DC' ORDER BY a.Title LIMIT 3;` is invalid because it's missing the table name for the `Name` column. Assuming that the `tracks`, `albums`, and `artists` tables have a common column with the artist names, we can modify the query to select from one of those tables.

2. The error "ambiguous column name: Name" occurs because both `t.Name` and `ar.Name` are referring to columns named 'Name'. Since `ar.Name` is likely the correct column to use (since it's referencing the artist's table), we should select from that table.

Here's a corrected SQL statement:

```sql
SELECT ar.Name FROM artists AS ar 
INNER JOIN albums AS a ON ar.ArtistId = a.ArtistId 
WHERE ar.Name = 'AC/DC' ORDER BY a.Title LIMIT 3;
```

However, this query will return the

In [34]:
print(response.metadata["sql_query"])

SELECT Name FROM tracks AS t INNER JOIN albums AS a ON t.AlbumId = a.AlbumId INNER JOIN artists AS ar ON a.ArtistId = ar.ArtistId WHERE ar.Name = 'AC/DC' ORDER BY a.Title LIMIT 3;


## 5. Structured Data Extraction

An important use case for function calling is extracting structured objects. LlamaIndex provides an intuitive interface for this through `structured_predict` - simply define the target Pydantic class (can be nested), and given a prompt, we extract out the desired object.

**NOTE**: Since there's no native function calling support with Llama3 / Ollama, the structured extraction is performed by prompting the LLM + output parsing.

In [35]:
from llama_index.llms.ollama import Ollama
from llama_index.core.prompts import PromptTemplate
from pydantic import BaseModel


class Restaurant(BaseModel):
    """A restaurant with name, city, and cuisine."""

    name: str
    city: str
    cuisine: str

llm = llm

prompt_tmpl = PromptTemplate(
    "Generate a restaurant in a given city {city_name}"
)

In [36]:
restaurant_obj = llm.structured_predict(
    Restaurant, prompt_tmpl, city_name="Miami"
)
print(restaurant_obj)

name='Coconut Coast' city='Miami' cuisine='Cuban-Seafood Fusion'


## 6. Adding Chat History to RAG (Chat Engine)

In this section we create a stateful chatbot from a RAG pipeline, with our chat engine abstraction.

Unlike a stateless query engine, the chat engine maintains conversation history (through a memory module like buffer memory). It performs retrieval given a condensed question, and feeds the condensed question + context + chat history into the final LLM prompt.

Related resource: https://docs.llamaindex.ai/en/stable/examples/chat_engine/chat_engine_condense_plus_context/

In [37]:
from llama_index.core.memory import ChatMemoryBuffer
from llama_index.core.chat_engine import CondensePlusContextChatEngine

memory = ChatMemoryBuffer.from_defaults(token_limit=3900)

chat_engine = CondensePlusContextChatEngine.from_defaults(
    index.as_retriever(),
    memory=memory,
    llm=llm,
    context_prompt=(
        "You are a chatbot, able to have normal interactions, as well as talk"
        " about the Kendrick and Drake beef."
        "Here are the relevant documents for the context:\n"
        "{context_str}"
        "\nInstruction: Use the previous chat history, or the context above, to interact and help the user."
    ),
    verbose=True,
)

In [38]:
response = chat_engine.chat(
    "Tell me about the songs Drake released in the beef."
)
print(str(response))

Condensed question: Tell me about the songs Drake released in the beef.
The infamous Kendrick Lamar/Drake beef. In this exchange, I can recall that Drake released two tracks as part of the beef.

Firstly, on May 5, he dropped "The Heart Part 6". This song is a response to Kendrick's previous diss, and it features a soulful beat with an Aretha Franklin sample. The lyrics are quite direct, with Drake countering Kendrick's accusations and implying that he's had enough of the beef.

Secondly, I couldn't find any information on a second track released by Drake specifically as part of this beef. However, it's worth noting that there is a song called "The Heart Part 1" (released in 2018) and another one called "Heart Part 2" (released in 2023), both of which seem to be related to the beef.

Would you like me to elaborate on any specific aspect of these songs or the beef itself?


In [39]:
response = chat_engine.chat("What about Kendrick?")
print(str(response))

Condensed question: Here is the rephrased follow-up question as a standalone inquiry:

What information can you provide about Kendrick Lamar's side of the beef?
Kendrick Lamar! In this beef, he's been quite active. Here are some of the tracks that come to mind:

Firstly, it was reported that Kendrick released a diss track called "N95" (also known as "Black Panther"-themed song with lyrics and sound design similar to "N95"). However, I couldn't find any confirmation on this track officially being released.

However, there is a well-known diss track by Pusha T targeting Drake, but that's not what we're looking for here. 

Then, Kendrick did release another track called “N95" was later followed by the release of "Black Panther"-themed song with lyrics and sound design similar to "N95", which some interpret as a follow-up to his earlier diss.

Additionally, there is a well-known track called "Not Like Us". In this song, Kendrick seems to be addressing Drake directly, with lyrics that are q

## 7. Agents

Here we build agents with Llama 3. We perform RAG over simple functions as well as the documents above.

### Agents And Tools

In [40]:
import json
from typing import Sequence, List

from llama_index.core.llms import ChatMessage
from llama_index.core.tools import BaseTool, FunctionTool
from llama_index.core.agent import ReActAgent

import nest_asyncio

nest_asyncio.apply()

### Define Tools

In [41]:
def multiply(a: int, b: int) -> int:
    """Multiple two integers and returns the result integer"""
    return a * b


def add(a: int, b: int) -> int:
    """Add two integers and returns the result integer"""
    return a + b


def subtract(a: int, b: int) -> int:
    """Subtract two integers and returns the result integer"""
    return a - b


def divide(a: int, b: int) -> int:
    """Divides two integers and returns the result integer"""
    return a / b


multiply_tool = FunctionTool.from_defaults(fn=multiply)
add_tool = FunctionTool.from_defaults(fn=add)
subtract_tool = FunctionTool.from_defaults(fn=subtract)
divide_tool = FunctionTool.from_defaults(fn=divide)

### ReAct Agent

In [42]:
agent = ReActAgent.from_tools(
    [multiply_tool, add_tool, subtract_tool, divide_tool],
    llm=llm,
    verbose=True,
)

### Querying

In [43]:
response = agent.chat("What is (121 + 2) * 5?")
print(str(response))

> Running step 9981b205-d8fc-4878-8081-2e9f6dc08ecd. Step input: What is (121 + 2) * 5?
Thought: The current language of the user is: English. I need to use a tool to help me answer the question.
Action: add
Action Input: {'a': 121, 'b': 2}
Observation: 123
> Running step 2fb7557e-1e56-47d8-ba44-501001bedb8f. Step input: None
Thought: The result of the multiplication operation after performing addition is still needed. I need to use another tool to complete the task.
Action: multiply
Action Input: {'a': 123, 'b': 5}
Observation: 615
> Running step b2954c50-3e31-4b72-ae11-2bdde6535290. Step input: None
Thought: The calculation is now complete. I can answer without using any more tools. I'll use the user's language to answer
Answer: The result of (121 + 2) * 5 is 615.
The result of (121 + 2) * 5 is 615.


### ReAct Agent With RAG QueryEngine Tools

In [44]:
from llama_index.core import (
    SimpleDirectoryReader,
    VectorStoreIndex,
    StorageContext,
    load_index_from_storage,
)

from llama_index.core.tools import QueryEngineTool, ToolMetadata
from llama_index.core.agent import ReActAgent

### Create ReAct Agent using RAG QueryEngine Tools

In [45]:
# Define query tools

drake_tool = QueryEngineTool(
    drake_index.as_query_engine(),
    metadata=ToolMetadata(
        name="drake_search",
        description="Useful for searching over Drake's life.",
    ),
)

kendrick_tool = QueryEngineTool(
    kendrick_index.as_query_engine(),
    metadata=ToolMetadata(
        name="kendrick_search",
        description="Useful for searching over Kendrick's life.",
    ),
)

query_engine_tools = [drake_tool, kendrick_tool]

In [46]:
from llama_index.core.agent import ReActAgent
from llama_index.core.prompts import PromptTemplate

# Force Ollama to follow ReAct format
react_prompt = PromptTemplate(
    "You are an AI assistant following the ReAct framework.\n"
    "For each input, you must respond in the following structured format:\n"
    "\n"
    "Thought: [Brief thought about the query]\n"
    "Action: [Choose a tool: 'drake_search' or 'kendrick_search']\n"
    "Input: [Relevant input for the chosen tool]\n"
    "\n"
    "If no external tool is needed, respond in this format:\n"
    "Thought: I can answer directly.\n"
    "Answer: [Your Answer Here]\n"
    "\n"
    "Now, respond to the following prompt:\n"
    "{query}"
)

# Update ReActAgent to use the correct format
agent = ReActAgent.from_tools(
    query_engine_tools,  
    llm=llm,  # Use Ollama
    prompt=react_prompt,  
    verbose=True,
)

### Querying

In [47]:
# Run the chat query
response = agent.chat("Tell me about how Kendrick and Drake grew up")
print(str(response))

> Running step 30bc52cd-e4a2-4ee4-adcf-34af97f77c4d. Step input: Tell me about how Kendrick and Drake grew up
Thought: The current language of the user is: English. I need to use a tool to help me answer the question.
Action: kendrick_search
Action Input: {'input': "Kendrick Lamar's childhood", 'location': 'compton'}
Observation: Kendrick Lamar grew up in a tough environment, facing hardships such as living in Section 8 housing, relying on welfare and food stamps, and experiencing homelessness. Despite these challenges, he found solace in music, particularly hip hop, which was influenced by his exposure to police brutality and the violent events that surrounded him during his childhood. His early life experiences had a profound impact on his interests and passions, shaping him into the person he became.
> Running step 549a583d-b3e7-497e-8a7e-cd086eb3bb70. Step input: None
Thought: I can answer without using any more tools. I'll use the user's language to answer
Answer: Kendrick Lamar g